In [ ]:
# Task 1 vs Task 2 性能对比散点图
# X轴：Task 1 SCS (Vision Capability)
# Y轴：Task 2 XDRFR (Editing Fidelity)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置图表样式
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'seaborn-darkgrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

# 数据根目录
# Resolve VCG-Bench root from the current notebook working directory.
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'src').exists():
            return path
    return start

BASE_DIR = find_repo_root()
DATA_DIR = BASE_DIR / 'data'
ANALYSIS_DIR = DATA_DIR / 'analysis'

# 创建必要的目录结构
(ANALYSIS_DIR / 'task12_analysis' / 'figures').mkdir(parents=True, exist_ok=True)

# 读取Task1和Task2的按模型统计数据
task1_by_model_path = ANALYSIS_DIR / 'task1_analysis' / 'tables' / 'task1_by_model.csv'
task2_by_model_path = ANALYSIS_DIR / 'task2_analysis' / 'tables' / 'task2_by_model.csv'

task1_df = pd.read_csv(task1_by_model_path)
task2_df = pd.read_csv(task2_by_model_path)

print(f"Task1模型数: {len(task1_df)}")
print(f"Task2模型数: {len(task2_df)}")

# 合并数据
merged_df = task1_df[['model', 'style_consistency_score_mean']].merge(
    task2_df[['model', 'xdrfr_mean']],
    on='model',
    how='inner'
)

print(f"\n合并后模型数: {len(merged_df)}")
print(f"\n合并数据预览:")
print(merged_df)

# 定义模型系列分类函数
def get_model_family(model_name):
    """根据模型名称返回模型系列"""
    model_lower = model_name.lower()
    if 'gpt' in model_lower:
        return 'GPT'
    elif 'claude' in model_lower:
        return 'Claude'
    elif 'gemini' in model_lower:
        return 'Gemini'
    elif 'qwen' in model_lower:
        return 'Qwen'
    else:
        return 'Other'

# 为每个模型添加系列标签
merged_df['model_family'] = merged_df['model'].apply(get_model_family)

# 定义颜色和标记
family_colors = {
    'GPT': '#10A37F',      # OpenAI绿色
    'Claude': '#FF6B35',   # Anthropic橙色
    'Gemini': '#4285F4',   # Google蓝色
    'Qwen': '#E53E3E',     # 红色
    'Other': '#808080'     # 灰色
}

family_markers = {
    'GPT': 'o',
    'Claude': 's',
    'Gemini': '^',
    'Qwen': '*',
    'Other': 'D'
}

# 创建散点图
fig, ax = plt.subplots(figsize=(10, 8))

# 按系列绘制散点
for family in merged_df['model_family'].unique():
    family_data = merged_df[merged_df['model_family'] == family]
    ax.scatter(family_data['style_consistency_score_mean'], 
               family_data['xdrfr_mean'],
               s=200, alpha=0.7, 
               color=family_colors[family],
               marker=family_markers[family],
               edgecolors='black', linewidth=1.5,
               label=family, zorder=3)

# 添加模型标签（简化名称）
for idx, row in merged_df.iterrows():
    model_name = row['model']
    # 简化模型名称
    if '/' in model_name:
        short_name = model_name.split('/')[-1]
    else:
        short_name = model_name
    
    # 进一步简化
    if 'Qwen3-VL-32B' in short_name:
        short_name = 'Qwen-32B'
    elif 'Qwen3-VL-235B' in short_name:
        short_name = 'Qwen-235B'
    elif 'gemini-3-pro' in short_name:
        short_name = 'Gemini-Pro'
    elif 'gemini-3-flash' in short_name:
        short_name = 'Gemini-Flash'
    elif 'claude-opus' in short_name:
        short_name = 'Claude-Opus'
    elif 'claude-sonnet' in short_name:
        short_name = 'Claude-Sonnet'
    elif 'gpt-5.2' in short_name:
        short_name = 'GPT-5.2'
    
    # 为不同模型设置不同的标签偏移，避免重叠
    if 'claude-opus' in model_name.lower():
        # Claude-Opus: 放在方块上面
        xytext_offset = (0, 15)
    elif 'claude-sonnet' in model_name.lower():
        # Claude-Sonnet: 放在方块下面
        xytext_offset = (0, -15)
    elif 'gemini-3-flash' in model_name.lower():
        # Gemini-Flash: 放在下面那个三角形下面
        xytext_offset = (0, -15)
    elif 'gemini-3-pro' in model_name.lower():
        # Gemini-Pro: 放在上面那个三角形上面
        xytext_offset = (0, 15)
    else:
        # 其他模型：默认偏移
        xytext_offset = (5, 5)
    
    ax.annotate(short_name, 
               (row['style_consistency_score_mean'], row['xdrfr_mean']),
               xytext=xytext_offset, textcoords='offset points',
               fontsize=9, alpha=0.8,
               ha='center' if 'claude' in model_name.lower() or 'gemini' in model_name.lower() else 'left')

# 添加"The Blind Expert Zone"标注（覆盖两个Qwen模型）
# 找到两个Qwen模型的位置
qwen_models = merged_df[merged_df['model'].str.contains('Qwen', case=False, na=False)]
if len(qwen_models) > 0:
    # 计算覆盖两个Qwen模型的矩形区域
    qwen_x_min = qwen_models['style_consistency_score_mean'].min()
    qwen_x_max = qwen_models['style_consistency_score_mean'].max()
    qwen_y_min = qwen_models['xdrfr_mean'].min()
    qwen_y_max = qwen_models['xdrfr_mean'].max()
    
    # 添加一些边距
    x_margin = (qwen_x_max - qwen_x_min) * 0.3 if qwen_x_max > qwen_x_min else 0.05
    y_margin = (qwen_y_max - qwen_y_min) * 0.3 if qwen_y_max > qwen_y_min else 0.02
    
    rect_x = qwen_x_min - x_margin
    rect_y = qwen_y_min - y_margin
    rect_width = (qwen_x_max - qwen_x_min) + 2 * x_margin if qwen_x_max > qwen_x_min else 0.15
    rect_height = (qwen_y_max - qwen_y_min) + 2 * y_margin if qwen_y_max > qwen_y_min else 0.05
    
    # 绘制黄色矩形区域
    rect = plt.Rectangle((rect_x, rect_y), rect_width, rect_height,
                        facecolor='yellow', alpha=0.3, edgecolor='red', 
                        linewidth=2, linestyle='--', zorder=0)
    ax.add_patch(rect)
    
    # 在区域上方添加文字标注
    ax.text(rect_x + rect_width / 2, rect_y + rect_height + 0.01, 
            "The Blind Expert Zone", 
            fontsize=11, fontweight='bold', style='italic',
            ha='center', va='bottom', color='darkred', zorder=5)

# 设置图表属性
ax.set_xlabel('Task 1 SCS (Vision Capability)', fontsize=13, fontweight='bold')
ax.set_ylabel('Task 2 XDRFR (Editing Fidelity)', fontsize=13, fontweight='bold')
ax.set_title('Vision Capability vs Editing Fidelity', fontsize=15, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, linestyle='--')

# 设置坐标轴范围
ax.set_xlim([-0.05, max(merged_df['style_consistency_score_mean']) * 1.15])
ax.set_ylim([0.5, max(merged_df['xdrfr_mean']) * 1.05])

# 添加图例
ax.legend(loc='lower right', fontsize=10, framealpha=0.9, title='Model Family', title_fontsize=11)

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'task12_analysis' / 'figures' / 'task12_vision_vs_editing.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"\n已保存图片到: {output_path}")

plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 尝试导入自动调整文字位置的库
try:
    from adjustText import adjust_text
    HAS_ADJUST_TEXT = True
except ImportError:
    HAS_ADJUST_TEXT = False
    print("提示: 建议 pip install adjustText 以获得最佳的文字排版效果")

# ---------------------------------------------------------
# 1. 基础设置
# ---------------------------------------------------------

# 设置中文字体 (兼容 Mac/Windows/Linux)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False

# 设置图表样式
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'seaborn-darkgrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

# ---------------------------------------------------------
# 2. 数据读取 (路径保持你的原设置)
# ---------------------------------------------------------
# Resolve VCG-Bench root from the current notebook working directory.
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'src').exists():
            return path
    return start

BASE_DIR = find_repo_root()
DATA_DIR = BASE_DIR / 'data'
ANALYSIS_DIR = DATA_DIR / 'analysis'

# 创建输出目录
output_dir = ANALYSIS_DIR / 'task12_analysis' / 'figures'
output_dir.mkdir(parents=True, exist_ok=True)

# 读取文件
task1_path = ANALYSIS_DIR / 'task1_analysis' / 'tables' / 'task1_by_model.csv'
task2_path = ANALYSIS_DIR / 'task2_analysis' / 'tables' / 'task2_by_model.csv'

print(f"正在读取数据...")
task1_df = pd.read_csv(task1_path)
task2_df = pd.read_csv(task2_path)

# 合并数据
merged_df = task1_df[['model', 'style_consistency_score_mean']].merge(
    task2_df[['model', 'xdrfr_mean']],
    on='model',
    how='inner'
)
print(f"合并后共有 {len(merged_df)} 个模型")

# 简化模型名称函数 (去除路径和多余前缀)
def clean_model_name(name):
    short = name.split('/')[-1]
    short = short.replace('Instruct', '').replace('preview', '').replace('claude-', '').strip('-')
    return short

merged_df['short_name'] = merged_df['model'].apply(clean_model_name)

# ---------------------------------------------------------
# 3. 绘图逻辑
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 9))

# 定义不同类别的模型
qwen_mask = merged_df['model'].str.contains('Qwen3-VL-32B|32B', case=False)
gemini_mask = merged_df['model'].str.contains('gemini', case=False)
other_mask = ~(qwen_mask | gemini_mask)

# 用于收集所有文本对象以便自动调整位置
texts = []

# --- A. 绘制普通模型 (灰色背景点) ---
others_df = merged_df[other_mask]
ax.scatter(others_df['style_consistency_score_mean'], 
           others_df['xdrfr_mean'],
           s=120, color='#9E9E9E', alpha=0.6, 
           edgecolors='white', linewidth=1, label='Other Models')

for _, row in others_df.iterrows():
    t = ax.text(row['style_consistency_score_mean'], row['xdrfr_mean'], 
                row['short_name'], fontsize=9, color='#616161', alpha=0.8)
    texts.append(t)

# --- B. 绘制 Qwen-32B (红色五角星 - Blind Expert) ---
qwen_df = merged_df[qwen_mask]
ax.scatter(qwen_df['style_consistency_score_mean'], 
           qwen_df['xdrfr_mean'],
           s=350, color='#D32F2F', marker='*', 
           edgecolors='white', linewidth=1.5, zorder=10, 
           label='Qwen-32B (Blind Expert)')

for _, row in qwen_df.iterrows():
    t = ax.text(row['style_consistency_score_mean'], row['xdrfr_mean'], 
                row['short_name'], fontsize=11, fontweight='bold', color='#B71C1C')
    texts.append(t)

# --- C. 绘制 Gemini (蓝色菱形 - SOTA) ---
gemini_df = merged_df[gemini_mask]
ax.scatter(gemini_df['style_consistency_score_mean'], 
           gemini_df['xdrfr_mean'],
           s=250, color='#1976D2', marker='D', 
           edgecolors='white', linewidth=1.5, zorder=10, 
           label='Gemini Series')

for _, row in gemini_df.iterrows():
    t = ax.text(row['style_consistency_score_mean'], row['xdrfr_mean'], 
                row['short_name'], fontsize=11, fontweight='bold', color='#0D47A1')
    texts.append(t)

# ---------------------------------------------------------
# 4. 关键：调整坐标轴范围 (Zoom In)
# ---------------------------------------------------------
# 获取数据的极值
y_min, y_max = merged_df['xdrfr_mean'].min(), merged_df['xdrfr_mean'].max()
x_min, x_max = merged_df['style_consistency_score_mean'].min(), merged_df['style_consistency_score_mean'].max()

# 计算 Y 轴的显示范围 (上下各留 15% 的余量)
y_span = y_max - y_min
y_lower_lim = y_min - (y_span * 0.5) # 底部留多点给 Blind Expert 标签
y_upper_lim = y_max + (y_span * 0.2)

ax.set_ylim([y_lower_lim, y_upper_lim])
ax.set_xlim([-0.05, x_max * 1.15]) # 左侧从 -0.05 开始，确保 0 分的点不贴边

# ---------------------------------------------------------
# 5. 添加区域标注 (The Blind Expert Zone)
# ---------------------------------------------------------
# 在左上角添加标注框
import matplotlib.patches as patches

# 这里手动指定一个区域，覆盖 Qwen 所在位置
# 如果 Qwen 分数变了，可能需要微调这里
rect_x = -0.03
rect_y = y_min - (y_span * 0.1)
rect_w = 0.15
rect_h = (y_max - rect_y) * 0.5

# 绘制虚线框
rect = patches.FancyBboxPatch((rect_x, rect_y), rect_w, rect_h,
                             boxstyle="round,pad=0.02", 
                             ec="#FFC107", fc="#FFF8E1", 
                             alpha=0.5, linestyle='--', linewidth=2, zorder=1)
ax.add_patch(rect)

ax.text(rect_x + 0.01, rect_y + rect_h + 0.002, "The Blind Expert Zone\n(Low Vision / High Edit)", 
        color='#FF8F00', fontsize=10, fontweight='bold', style='italic', va='bottom')

# ---------------------------------------------------------
# 6. 美化与保存
# ---------------------------------------------------------
ax.set_xlabel('Task 1: Vision Capability (SCS)', fontsize=13, fontweight='bold', labelpad=10)
ax.set_ylabel('Task 2: Editing Fidelity (XDRFR)', fontsize=13, fontweight='bold', labelpad=10)
ax.set_title('Vision Capability vs Editing Fidelity', fontsize=16, fontweight='bold', pad=25)

# 细网格
ax.minorticks_on()
ax.grid(True, which='major', linestyle='-', linewidth=0.8, alpha=0.8)
ax.grid(True, which='minor', linestyle=':', linewidth=0.5, alpha=0.4)

# 图例
ax.legend(loc='lower right', frameon=True, framealpha=0.95, fontsize=10, shadow=True, borderpad=1)

# 自动调整文字位置 (防止重叠)
if HAS_ADJUST_TEXT:
    print("正在优化文字布局...")
    adjust_text(texts, 
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5, alpha=0.5),
                expand_points=(1.5, 1.5))

plt.tight_layout()

# 保存
output_path = output_dir / 'task12_vision_vs_editing_optimized.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"\n图片已保存至: {output_path}")

plt.show()